# Частина 1
## Завдання 1: Завантаження даних VHI за допомогою urllib

In [40]:
import urllib.request
import datetime
import os

def download_vhi_data():
    if not os.path.exists('data'):
        os.makedirs('data')

    for i in range(1, 28):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"

        now = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"data/vhi_id_{i}_{now}.csv"

        existing_files = [f for f in os.listdir('data') if f.startswith(f"vhi_id_{i}_")]
        if existing_files:
            print(f"Файл для ID {i} вже існує. Пропускаємо.")
            continue
            
        try:
            with urllib.request.urlopen(url) as response:
                content = response.read().decode('utf-8')
                with open(filename, 'w') as f:
                    f.write(content)
            print(f"ID {i} завантажено успішно.")
        except Exception as e:
            print(f"Помилка при завантаженні ID {i}: {e}")

download_vhi_data()

Файл для ID 1 вже існує. Пропускаємо.
Файл для ID 2 вже існує. Пропускаємо.
Файл для ID 3 вже існує. Пропускаємо.
Файл для ID 4 вже існує. Пропускаємо.
Файл для ID 5 вже існує. Пропускаємо.
Файл для ID 6 вже існує. Пропускаємо.
Файл для ID 7 вже існує. Пропускаємо.
Файл для ID 8 вже існує. Пропускаємо.
Файл для ID 9 вже існує. Пропускаємо.
Файл для ID 10 вже існує. Пропускаємо.
Файл для ID 11 вже існує. Пропускаємо.
Файл для ID 12 вже існує. Пропускаємо.
Файл для ID 13 вже існує. Пропускаємо.
Файл для ID 14 вже існує. Пропускаємо.
Файл для ID 15 вже існує. Пропускаємо.
Файл для ID 16 вже існує. Пропускаємо.
Файл для ID 17 вже існує. Пропускаємо.
Файл для ID 18 вже існує. Пропускаємо.
Файл для ID 19 вже існує. Пропускаємо.
Файл для ID 20 вже існує. Пропускаємо.
Файл для ID 21 вже існує. Пропускаємо.
Файл для ID 22 вже існує. Пропускаємо.
Файл для ID 23 вже існує. Пропускаємо.
Файл для ID 24 вже існує. Пропускаємо.
Файл для ID 25 вже існує. Пропускаємо.
Файл для ID 26 вже існує. Пропуска

## Завдання 2: Зчитування у Pandas DataFrame та Data Cleaning

In [48]:
import pandas as pd
import glob
import os
import io

def create_df(directory):
    path = os.path.join(directory, "*.csv")
    all_files = glob.glob(path)
    
    if not all_files:
        print(f"Помилка: Файлів у '{directory}' не знайдено.")
        return pd.DataFrame()

    all_data = []

    noaa_to_name = {
        1: "Черкаська", 2: "Чернігівська", 3: "Чернівецька", 4: "Крим", 5: "Дніпропетровська",
        6: "Донецька", 7: "Івано-Франківська", 8: "Харківська", 9: "Херсонська", 10: "Хмельницька",
        11: "Київська", 12: "Кіровоградська", 13: "Луганська", 14: "Львівська", 15: "Миколаївська",
        16: "Одеська", 17: "Полтавська", 18: "Рівненська", 19: "Севастополь", 20: "Сумська",
        21: "Тернопільська", 22: "Закарпатська", 23: "Вінницька", 24: "Волинська", 25: "Запорізька",
        26: "Житомирська", 27: "Київ"
    }

    ordered_provinces = [
        "Вінницька", "Волинська", "Дніпропетровська", "Донецька", "Житомирська",
        "Закарпатська", "Запорізька", "Івано-Франківська", "Київська", "Кіровоградська",
        "Луганська", "Львівська", "Миколаївська", "Одеська", "Полтавська",
        "Рівненська", "Сумська", "Тернопільська", "Харківська", "Херсонська",
        "Хмельницька", "Черкаська", "Чернівецька", "Чернігівська", 
        "Крим", "Київ", "Севастополь"
    ]

    ua_index_map = {name: i+1 for i, name in enumerate(ordered_provinces)}

    for file in all_files:
        try:
            with open(file, 'r') as f:
                lines = f.readlines()
            
            clean_rows = []
            for line in lines:
                line = line.replace('<pre>', '').replace('</pre>', '').replace('<br>', '').strip()
                
                if not line or 'year' in line.lower():
                    continue
                
                parts = line.split(',')
                if len(parts) >= 7:
                    clean_rows.append(parts[:7])

            temp_df = pd.DataFrame(clean_rows, columns=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])

            for col in temp_df.columns:
                temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce')

            temp_df = temp_df.dropna(subset=['Year', 'VHI'])
            temp_df = temp_df[temp_df['VHI'] != -1]

            orig_id = int(os.path.basename(file).split('_')[2])
            p_name = noaa_to_name.get(orig_id, "Unknown")
            
            temp_df['Province'] = p_name
            temp_df['UA_ID'] = ua_index_map.get(p_name, 0)
            
            all_data.append(temp_df)
            
        except Exception as e:
            print(f"Помилка у файлі {file}: {e}")

    if not all_data:
        return pd.DataFrame()

    return pd.concat(all_data, ignore_index=True)

df = create_df('data')

if not df.empty:
    print(f"Успішно завантажено! Рядки: {len(df)}")
    vinn_id = df[df['Province']=='Вінницька']['UA_ID'].iloc[0]
    print(f"Приклад даних (Вінницька область має ID {vinn_id}):")
    print(df.head())

Успішно завантажено! Рядки: 58995
Приклад даних (Вінницька область має ID 1):
     Year  Week    SMN     SMT    VCI    TCI    VHI     Province  UA_ID
0  1982.0     2  0.063  261.53  55.89  38.20  47.04  Хмельницька     21
1  1982.0     3  0.063  263.45  57.30  32.69  44.99  Хмельницька     21
2  1982.0     4  0.061  265.10  53.96  28.62  41.29  Хмельницька     21
3  1982.0     5  0.058  266.42  46.87  28.57  37.72  Хмельницька     21
4  1982.0     6  0.056  267.47  39.55  30.27  34.91  Хмельницька     21


1. Ряд VHI для області за вказаний рік

In [49]:
def get_vhi_by_year(df, ua_id, year):
    result = df[(df['UA_ID'] == ua_id) & (df['Year'] == year)][['Week', 'VHI', 'Province']]
    
    if result.empty:
        print(f"Даних для ID {ua_id} за {year} рік не знайдено.")
        return None
    
    province_name = result['Province'].iloc[0]
    print(f"--- Ряд VHI для області: {province_name} (ID: {ua_id}) за {year} рік ---")
    return result
get_vhi_by_year(df, 3, 2023)

--- Ряд VHI для області: Дніпропетровська (ID: 3) за 2023 рік ---


,Week,VHI,Province
50151,1,49.77,Дніпропетровська
50152,2,48.97,Дніпропетровська
50153,3,48.45,Дніпропетровська
50154,4,48.09,Дніпропетровська
50155,5,46.84,Дніпропетровська
50156,6,45.37,Дніпропетровська
50157,7,44.30,Дніпропетровська
50158,8,44.29,Дніпропетровська
50159,9,44.71,Дніпропетровська
50160,10,43.87,Дніпропетровська


2. Ряд VHI за діапазон років для вказаних областей

In [50]:
def get_vhi_range(df, ua_ids, start_year, end_year):
    result = df[(df['UA_ID'].isin(ua_ids)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]
    
    if result.empty:
        print("За вказаними параметрами даних не знайдено.")
        return None
    
    print(f"--- Дані VHI для областей {ua_ids} за період {int(start_year)}-{int(end_year)} ---")
    
    display_df = result[['Year', 'Week', 'Province', 'VHI', 'UA_ID']]

    return display_df.sort_values(by=['Year', 'UA_ID'])

get_vhi_range(df, [3, 10], 2020, 2022)

--- Дані VHI для областей [3, 10] за період 2020-2022 ---


,Year,Week,Province,VHI,UA_ID
49995,2020.0,1,Дніпропетровська,47.23,3
49996,2020.0,2,Дніпропетровська,46.99,3
49997,2020.0,3,Дніпропетровська,46.92,3
49998,2020.0,4,Дніпропетровська,46.55,3
49999,2020.0,5,Дніпропетровська,45.87,3
...,...,...,...,...,...
6446,2022.0,48,Кіровоградська,40.12,10
6447,2022.0,49,Кіровоградська,42.05,10
6448,2022.0,50,Кіровоградська,41.30,10
6449,2022.0,51,Кіровоградська,39.95,10


3. Пошук екстремумів, середнього та медіани

In [53]:
def get_vhi_statistics(df, ua_id, start_year, end_year):
    subset = df[(df['UA_ID'] == ua_id) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]['VHI']
    
    if subset.empty:
        print("Немає даних для розрахунку статистики.")
        return
    
    province_name = df[df['UA_ID'] == ua_id]['Province'].iloc[0]
    
    print(f"--- Статистика VHI: {province_name} ({int(start_year)}-{int(end_year)}) ---")
    stats = {
        "Мінімум (Min)": subset.min(),
        "Максимум (Max)": subset.max(),
        "Середнє (Mean)": round(subset.mean(), 2),
        "Медіана (Median)": subset.median()
    }
    
    for key, value in stats.items():
        print(f"{key}: {value}")

get_vhi_statistics(df, 24, 2015, 2024)

--- Статистика VHI: Чернігівська (2015-2024) ---
Мінімум (Min): 23.16
Максимум (Max): 78.8
Середнє (Mean): 46.63
Медіана (Median): 45.790000000000006
